# LearnMate — Deadline profile: reduced end-to-end run
Runs on a free Colab **T4** (Runtime → Change runtime type → GPU). Checkpoints and caches are written to Google Drive so a disconnect never loses work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone -b claude/wonderful-bohr-1idsaa https://github.com/KirtirajChaudhari/LearnMate.git || (cd LearnMate && git pull)
%cd /content/LearnMate

In [ ]:
!pip -q install -e '.[gpu,dev]'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# persist large/expensive artifacts on Drive (safe against directories that are tracked in git)
import os, shutil
D = '/content/drive/MyDrive/learnmate'
for dst, sub in {'data/raw': 'data_raw', 'data/processed': 'data_processed', 'data/cache': 'data_cache', 'models': 'models', 'results': 'results'}.items():
    os.makedirs(f'{D}/{sub}', exist_ok=True)
    if os.path.islink(dst): os.remove(dst)
    elif os.path.exists(dst): shutil.rmtree(dst)
    os.makedirs(os.path.dirname(dst) or '.', exist_ok=True)
    os.symlink(f'{D}/{sub}', dst)
print(os.listdir('/content/LearnMate/data'))

In [ ]:
!test -d data/raw/d2l-en || git clone https://github.com/d2l-ai/d2l-en.git data/raw/d2l-en
!git -C data/raw/d2l-en checkout -q 23d7a5aecceee57d1292c56e90cce307f183bb0a
!python -m learnmate.ingest --check
!python scripts/build_squad.py

**Run the parts in order. Every stage is cached and resumable: after a disconnect, rerun the same cell.**

Part 1 needs no human input.

## Part 1a — indices and the sufficiency gate

In [ ]:
!python scripts/build_indices.py

In [ ]:
!python -m learnmate.gate.train --out models/gate-v1 --pairs 24000 --epochs 1 --resume
!cat models/gate-v1/train_log.json | head -40

## Part 1b — calibration-split run (SQuAD, 300 questions) and calibration

In [ ]:
!python -m eval.run --split squad-cal --stages retrieve generate gate b0 verify assemble

In [ ]:
!python -m eval.calibrate --squad-cal results/runs/squad-cal__clean

## STOP — Part 2 needs the frozen course items (git pull the verified `data/annotations/course_items.jsonl`, then `python -m learnmate.annotation freeze`) and `python scripts/freeze_prereg.py`.

In [ ]:
!git pull -q --autostash
!python -m learnmate.annotation validate
!python -m learnmate.annotation assign-splits --all-split test
!python -m learnmate.annotation kappa
!python -m learnmate.annotation freeze
!python scripts/freeze_prereg.py --dtype float16

## Part 2 — locked test runs (once)

In [ ]:
!python -m eval.run --split course-test --condition clean --stages retrieve generate gate b0 verify assemble --require-prereg
!python -m eval.run --split course-test --condition S1 --stages retrieve generate gate b0 verify assemble --require-prereg
!python -m eval.run --split squad-test --stages retrieve generate gate b0 verify assemble --require-prereg

## Part 3 — blinded judging sheet (download `judge_test.jsonl`, judge with `python scripts/review.py judge`, upload the judged file back)

In [ ]:
!python -m learnmate.annotation judge-sheet --run results/runs/course-test__clean --out data/annotations/judge_test.jsonl

After judging: `python -m learnmate.annotation merge --sheet data/annotations/judge_test.jsonl --key data/annotations/judge_test_KEY_DO_NOT_OPEN.jsonl --out data/annotations/judgments_test.jsonl`

In [ ]:
!python -m learnmate.annotation merge --sheet data/annotations/judge_test.jsonl --key data/annotations/judge_test_KEY_DO_NOT_OPEN.jsonl --out data/annotations/judgments_test.jsonl
!python -m eval.analyze --run results/runs/course-test__clean --judgments data/annotations/judgments_test.jsonl
!python -m eval.analyze --run results/runs/course-test__S1
!python -m eval.analyze --run results/runs/squad-test__clean
!python scripts/make_figures.py

## Optional — live demo from Colab (for the recorded walkthrough)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, time
subprocess.Popen(['streamlit','run','app/streamlit_app.py','--server.port','8501','--server.headless','true'])
subprocess.Popen(['./cloudflared','tunnel','--url','http://localhost:8501'], stdout=open('tunnel.log','w'), stderr=subprocess.STDOUT)
time.sleep(15)
!grep -o 'https://[a-z0-9-]*[.]trycloudflare[.]com' tunnel.log | head -1